In [1]:
import numpy as np
import matplotlib.pyplot as plt
import cv2


KeyboardInterrupt: 

In [ ]:
topp,bottomm,rightt,leftt=[],[],[],[]
pp=[]


In [ ]:
image_path = r'C:\RC\Test\img1111.png'
image = cv2.imread(image_path)

gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray_image, (3,3), 0)

_, thresh_init = cv2.threshold(blurred, 200, 255, cv2.THRESH_BINARY) 

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
closed = cv2.morphologyEx(thresh_init, cv2.MORPH_CLOSE, kernel)
contours, hierarchy = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

min_area = 500 
valid_contours = [c for c in contours if cv2.contourArea(c) > min_area]

isolated_pieces_list = []  
for contour in valid_contours: 
    x, y, w, h = cv2.boundingRect(contour)
    pad = 0
    
    x_start, x_end = max(0, x - pad), min(image.shape[1], x + w + pad)
    y_start, y_end = max(0, y - pad), min(image.shape[0], y + h + pad)
    
    cropped_piece = image[y_start:y_end, x_start:x_end]
    
    mask = np.zeros_like(closed)
    cv2.drawContours(mask, [contour], -1, 255, -1) 

    cropped_mask = mask[y_start:y_end, x_start:x_end]
    isolated_piece = cv2.bitwise_and(cropped_piece, cropped_piece, mask=cropped_mask)
    
    isolated_pieces_list.append(isolated_piece)
ccc=-1
for  current_piece in isolated_pieces_list:
    padded_image = cv2.copyMakeBorder(current_piece, 10, 10, 10, 10, borderType=cv2.BORDER_CONSTANT, value=[0,0,0])
    
    gray = cv2.cvtColor(padded_image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    
    points = cv2.findNonZero(thresh)
    
    x_box, y_box, w_box, h_box = cv2.boundingRect(points)

    tol = 1.5 
    top_deviations, bottom_deviations = [], []

    for col in range(x_box, x_box + w_box):
        white_pixels = np.where(thresh[:, col] == 255)[0]
        if len(white_pixels) > 0:
            top_deviations.append(int(white_pixels[0] - y_box))
            bottom_deviations.append(int((y_box + h_box) - white_pixels[-1]))

    std_top = np.std(top_deviations) if top_deviations else 0
    std_bottom = np.std(bottom_deviations) if bottom_deviations else 0
    
    top_is_torn = std_top > tol
    bottom_is_torn = std_bottom > tol
    
    print(f"Top :    {'TORN' if top_is_torn else 'STRAIGHT'} (variance: {std_top:.2f}px)")
    print(f"Bottom : {'TORN' if bottom_is_torn else 'STRAIGHT'} (variance: {std_bottom:.2f}px)")

    left_avg, right_avg = [], []
    for row in range(y_box, y_box + h_box):
        white_pixels = np.where(thresh[row, :] == 255)[0]
        if len(white_pixels) > 0:
            left_avg.append(int(white_pixels[0] - x_box))
            right_avg.append(int((x_box + w_box) - white_pixels[-1]))

    std_left = np.std(left_avg) if left_avg else 0
    std_right = np.std(right_avg) if right_avg else 0
    
    left_is_torn = std_left > tol
    right_is_torn = std_right > tol
    
    print(f"Left :   {'TORN' if left_is_torn else 'STRAIGHT'} (avg: {std_left:.2f}px)")
    print(f"Right :  {'TORN' if right_is_torn else 'STRAIGHT'} (avg: {std_right:.2f}px)")

    a1 = cv2.Canny(thresh, 150, 150)
    y_arr, x_arr = np.where(a1 == 255)
    
    y_pixels, x_pixels = np.where(thresh == 255)
    min_y, max_y = np.min(y_pixels), np.max(y_pixels)
    min_x, max_x = np.min(x_pixels), np.max(x_pixels)
    
    g1 = 5  
    any_torn = False

    if top_is_torn:
        any_torn = True
        b11_top = []
        for col_x in range(min_x, max_x + 1):
            pixels_in_col = y_arr[x_arr == col_x]
            if len(pixels_in_col) > 0:
                b11_top.append((int(col_x), int(np.min(pixels_in_col))))
        
        top_ref_y = int(min_y - g1)
        distances_top = [abs(int(y11) - top_ref_y) for _, y11 in b11_top]
        print(f"TOP Distances{distances_top}")

        topp.append(distances_top)

        
    else:
        topp.append(np.nan)

    if bottom_is_torn:
        any_torn = True
        b11_bottom = []
        for col_x in range(min_x, max_x + 1):
            pixels_in_col = y_arr[x_arr == col_x]
            if len(pixels_in_col) > 0:
                b11_bottom.append((int(col_x), int(np.max(pixels_in_col))))
        
        bottom_ref_y = int(max_y + g1)
        distances_bottom = [abs(int(y11) - bottom_ref_y) for _, y11 in b11_bottom]
        print(f"BOTTOM Distances{distances_bottom}")
        
        bottomm.append(distances_bottom)
    else:
        bottomm.append(np.nan)

    if left_is_torn:
        any_torn = True
        b11_left = []
        for row_y in range(min_y, max_y + 1):
            pixels_in_row = x_arr[y_arr == row_y]
            if len(pixels_in_row) > 0:
                b11_left.append((int(row_y), int(np.min(pixels_in_row))))
        
        left_ref_x = int(min_x - g1)
        distances_left = [abs(int(x11) - left_ref_x) for _, x11 in b11_left]
        print(f"LEFT Distances {distances_left}")
       
        leftt.append(distances_left)
    else:
        leftt.append(np.nan)

    if right_is_torn:
        any_torn = True
        b11_right = []
        for row_y in range(min_y, max_y + 1):
            pixels_in_row = x_arr[y_arr == row_y]
            if len(pixels_in_row) > 0:
                b11_right.append((int(row_y), int(np.max(pixels_in_row))))
        
        right_ref_x = int(max_x + g1)
        distances_right = [abs(int(x11) - right_ref_x) for _, x11 in b11_right]
        
        print(f"RIGHT Distances{distances_right}")
      
        rightt.append(distances_right)
        

    else:
        rightt.append(np.nan)
    ccc+=1
    print(ccc)
    pp.append(thresh)
    plt.imshow(thresh, cmap='gray')
    plt.show()




In [ ]:
side1 = rightt[0]
side2 = leftt[1]
print(side1)
print(side2)
th = 0.82
tr = 12

edge1 = np.array(side1 if len(side1) > 2*tr else side1, dtype=float)
edge2 = np.array(side2 if len(side2) > 2*tr else side2, dtype=float)

x1 = np.arange(len(edge1))
slope1, intercept1 = np.polyfit(x1, edge1, 1)
detrended1 = edge1 - (slope1 * x1 + intercept1)
roughness1 = np.std(detrended1)

x2 = np.arange(len(edge2))
slope2, intercept2 = np.polyfit(x2, edge2, 1)
detrended2 = edge2 - (slope2 * x2 + intercept2)
roughness2 = np.std(detrended2)

if roughness1 < 1.2 or roughness2 < 1.2:
    print("NOT MATCHED ")
  

else:
    norm1 = (edge1 - np.min(edge1)) / (np.max(edge1) - np.min(edge1) + 1e-6)
    norm2 = (edge2 - np.min(edge2)) / (np.max(edge2) - np.min(edge2) + 1e-6)

    if len(norm1) < len(norm2):
        shorter, longer = norm1, norm2
        is1_shorter = True
    else:
        shorter, longer = norm2, norm1
        is1_shorter = False
        
    s_len, l_len = len(shorter), len(longer)

    best_score = 0.0
    best_offset = 0

    for direction in ["Forward", "Reversed"]:
        test_shorter = shorter if direction == "Forward" else shorter[::-1]
        
        for offset in range(l_len - s_len + 1):
            segment = longer[offset:offset + s_len]
            
            dist_direct = np.mean(np.abs(test_shorter - segment))
            score_direct = 1.0 - dist_direct
            
            dist_comp = np.mean(np.abs(test_shorter - (1.0 - segment)))
            score_comp = 1.0 - dist_comp
            
            if score_direct > best_score:
                best_score = score_direct
                best_offset = offset if is1_shorter else -offset
                
            if score_comp > best_score:
                best_score = score_comp
                best_offset = offset if is1_shorter else -offset

    confidence = best_score * 100

    if best_score >= th:
        print("MATCHED ")
        print(f"Confidence Score: {confidence:.2f}%")
    else:
        print("NOT MATCHED ")
        print(f"Confidence Score: {confidence:.2f}%")



In [ ]:
# def get_thresh(img):
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#     _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
#     return thresh

# def merge_canvases(imgA, imgB, threshA, threshB, dx, dy):
#     hA, wA = imgA.shape[:2]
#     hB, wB = imgB.shape[:2]
    
#     min_x, max_x = min(0, dx), max(wA, wB + dx)
#     min_y, max_y = min(0, dy), max(hA, hB + dy)
    
#     canvas = np.zeros((max_y - min_y, max_x - min_x, 3), dtype=np.uint8)
    
#     canvas[-min_y:-min_y+hA, -min_x:-min_x+wA][threshA > 0] = imgA[threshA > 0]
#     canvas[dy-min_y:dy-min_y+hB, dx-min_x:dx-min_x+wB][threshB > 0] = imgB[threshB > 0]
#     return canvas

# def stitch_by_edge(img2, img3, ref="bottom"):
#     thresh2, thresh3 = get_thresh(img2), get_thresh(img3)
    
#     p2, p3 = np.where(thresh2 == 255), np.where(thresh3 == 255)
#     dy = (int(p2[0].max()) - int(p3[0].max())) if ref == "bottom" else (int(p2[0].min()) - int(p3[0].min()))
    
#     y_arr2, x_arr2 = np.where(cv2.Canny(thresh2, 150, 150) == 255)
#     b2_coords = {r: np.max(x_arr2[y_arr2 == r]) for r in range(np.min(p2[0]), np.max(p2[0]) + 1) if len(x_arr2[y_arr2 == r]) > 0}
    
#     y_arr3, x_arr3 = np.where(cv2.Canny(thresh3, 150, 150) == 255)
#     b3_coords = {r: np.min(x_arr3[y_arr3 == r]) for r in range(np.min(p3[0]), np.max(p3[0]) + 1) if len(x_arr3[y_arr3 == r]) > 0}
    
#     dx_offsets = [b2_coords[y3 + dy] - x3 for y3, x3 in b3_coords.items() if (y3 + dy) in b2_coords]
#     dx = int(np.percentile(dx_offsets, 90))
    
#     return merge_canvases(img2, img3, thresh2, thresh3, dx, dy)

# def stitch_by_margin(img0, img1, side="left"):
#     thresh0, thresh1 = get_thresh(img0), get_thresh(img1)
    
#     y0, x0 = np.where(thresh0 == 255)
#     y1, x1 = np.where(thresh1 == 255)
    
#     edge0 = np.min(x0) if side == "left" else np.max(x0)
#     edge1 = np.min(x1) if side == "left" else np.max(x1)
#     dx = edge0 - edge1
    
#     bottom0 = {c: np.max(y0[x0 == c]) for c in np.unique(x0)}
#     top1 = {c + dx: np.min(y1[x1 == c]) for c in np.unique(x1)}
    
#     dy_offsets = [bottom0[c] - top1[c] + 1 for c in bottom0 if c in top1]
#     dy = int(np.percentile(dy_offsets, 92))
    
#     return merge_canvases(img0, img1, thresh0, thresh1, dx, dy)

# def stitch_right_column(img_left, img_right):
#     thresh_l, thresh_r = get_thresh(img_left), get_thresh(img_right)
#     p_l, p_r = np.where(thresh_l == 255), np.where(thresh_r == 255)
    
#     dy = int(p_l[0].max()) - int(p_r[0].max())
    
#     y_l, x_l = np.where(cv2.Canny(thresh_l, 150, 150) == 255)
#     b_l = {r: np.max(x_l[y_l == r]) for r in range(np.min(p_l[0]), np.max(p_l[0]) + 1) if len(x_l[y_l == r]) > 0}
    
#     y_r, x_r = np.where(cv2.Canny(thresh_r, 150, 150) == 255)
#     b_r = {r: np.min(x_r[y_r == r]) for r in range(np.min(p_r[0]), np.max(p_r[0]) + 1) if len(x_r[y_r == r]) > 0}
    
#     dx_offsets = [b_l[y3 + dy] - x3 for y3, x3 in b_r.items() if (y3 + dy) in b_l]
#     dx = int(np.percentile(dx_offsets, 45)) 
    
#     return merge_canvases(img_left, img_right, thresh_l, thresh_r, dx, dy)

# def ad_stitch(img_base, img_patch):
#     m_b = (cv2.cvtColor(img_base, cv2.COLOR_BGR2GRAY) > 15).astype(np.uint8) * 255
#     m_p = (cv2.cvtColor(img_patch, cv2.COLOR_BGR2GRAY) > 15).astype(np.uint8) * 255
    
#     y_p, x_p = np.where(cv2.Canny(m_p, 50, 150) == 255)
    
#     dist_out = cv2.distanceTransform(~m_b, cv2.DIST_L2, 5)
#     dist_in = cv2.distanceTransform(m_b, cv2.DIST_L2, 5)
    
#     h_b, w_b = img_base.shape[:2]
#     h_p, w_p = img_patch.shape[:2]
    
#     best_dx, best_dy, min_score = 0, 0, float('inf')
    
#     for dy in range(-int(h_b * 0.4), int(h_b * 0.8)):
#         for dx in range(int(w_b * 0.1), int(w_b * 1.2)):
#             sy, sx = y_p + dy, x_p + dx
#             valid = (sy >= 0) & (sy < h_b) & (sx >= 0) & (sx < w_b)
            
#             if np.sum(valid) < 15:
#                 continue
                
#             vy, vx = sy[valid], sx[valid]
#             ol = m_b[vy, vx] == 255
#             ol_cnt = np.sum(ol)
            
#             if ol_cnt / len(vy) > 0.35:
#                 continue
                
#             n_vy, n_vx = vy[~ol], vx[~ol]
#             if len(n_vy) == 0:
#                 continue
                
#             gaps = dist_out[n_vy, n_vx]
#             penalty = np.mean(dist_in[vy[ol], vx[ol]]) * 3.0 if ol_cnt > 0 else 0.0
#             score = np.mean(gaps) + penalty - np.sum(gaps < 4.0) * 0.2
            
#             if score < min_score:
#                 min_score, best_dx, best_dy = score, dx, dy

#     oy, ox = max(0, -best_dy), max(0, -best_dx)
#     mh = max(h_b + oy, h_p + best_dy + oy)
#     mw = max(w_b + ox, w_p + best_dx + ox)
    
#     canvas = np.zeros((mh, mw, 3), dtype=np.uint8)
#     canvas[oy:oy+h_b, ox:ox+w_b] = img_base
    
#     py, px = best_dy + oy, best_dx + ox
#     canvas[py:py+h_p, px:px+w_p][m_p == 255] = img_patch[m_p == 255]
    
#     return canvas

In [ ]:

# canvas = isolated_pieces_list[0]
# canvas = stitch_by_edge(canvas, isolated_pieces_list[1], ref="bottom")
# canvas = stitch_by_edge(canvas, isolated_pieces_list[2], ref="bottom")
# canvas = stitch_by_margin(isolated_pieces_list[4], canvas, side="left")
# canvas75 = stitch_by_edge(isolated_pieces_list[7], isolated_pieces_list[5], ref="bottom")
# canvas = stitch_by_margin(canvas75, canvas, side="left")
# canvas63 = stitch_by_margin(isolated_pieces_list[6], isolated_pieces_list[3], side="right")
# canvas638 = stitch_by_margin(isolated_pieces_list[8], canvas63, side="right")

# canvas = stitch_by_edge(canvas, canvas638, ref="bottom")
# canvas9_10 = stitch_by_edge(isolated_pieces_list[9], isolated_pieces_list[10], ref="bottom")
# canvas = stitch_by_margin(canvas9_10, canvas, side="left")
# canvas_12_11 = ad_stitch(isolated_pieces_list[12], isolated_pieces_list[11])
# canvas = stitch_by_margin(canvas_12_11, canvas, side="left")
# canvas = stitch_by_margin(isolated_pieces_list[15], canvas, side="left")
# canvas_19_16 = ad_stitch(isolated_pieces_list[19], isolated_pieces_list[16])

# canvas_13_14 = ad_stitch(isolated_pieces_list[13], isolated_pieces_list[14])
# c1= stitch_by_margin(isolated_pieces_list[17],canvas_13_14, side="right")
# c2 = stitch_by_edge(canvas_19_16, isolated_pieces_list[20], ref="top")
# c2 = stitch_by_edge(c2, isolated_pieces_list[18], ref="top")
# c2 = stitch_by_edge(c2,c1, ref="top")
# canvas = stitch_by_margin(c2, canvas, side="left")
# plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
# plt.show()

In [ ]:
def get_thresh(img):
    _, thresh = cv2.threshold(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 15, 255, cv2.THRESH_BINARY)
    return thresh

def merge_canvases(imgA, imgB, threshA, threshB, dx, dy):
    hA, wA = imgA.shape[:2]
    hB, wB = imgB.shape[:2]
    min_x, max_x = min(0, dx), max(wA, wB + dx)
    min_y, max_y = min(0, dy), max(hA, hB + dy)
    
    canvas = np.zeros((max_y - min_y, max_x - min_x, 3), dtype=np.uint8)
    canvas[-min_y:-min_y+hA, -min_x:-min_x+wA][threshA > 0] = imgA[threshA > 0]
    canvas[dy-min_y:dy-min_y+hB, dx-min_x:dx-min_x+wB][threshB > 0] = imgB[threshB > 0]
    return canvas

def stitch_by_edge(img2, img3, ref="bottom"):
    thresh2, thresh3 = get_thresh(img2), get_thresh(img3)
    p2, p3 = np.where(thresh2 == 255), np.where(thresh3 == 255)
    dy = (int(p2[0].max()) - int(p3[0].max())) if ref == "bottom" else (int(p2[0].min()) - int(p3[0].min()))
    
    y2, x2 = np.where(cv2.Canny(thresh2, 150, 150) == 255)
    b2 = {r: np.max(x2[y2 == r]) for r in range(np.min(p2[0]), np.max(p2[0]) + 1) if len(x2[y2 == r]) > 0}
    
    y3, x3 = np.where(cv2.Canny(thresh3, 150, 150) == 255)
    b3 = {r: np.min(x3[y3 == r]) for r in range(np.min(p3[0]), np.max(p3[0]) + 1) if len(x3[y3 == r]) > 0}
    
    dx = int(np.percentile([b2[y + dy] - x for y, x in b3.items() if (y + dy) in b2], 90))
    return merge_canvases(img2, img3, thresh2, thresh3, dx, dy)

def stitch_by_margin(img0, img1, side="left"):
    thresh0, thresh1 = get_thresh(img0), get_thresh(img1)
    y0, x0 = np.where(thresh0 == 255)
    y1, x1 = np.where(thresh1 == 255)
    
    dx = (np.min(x0) if side == "left" else np.max(x0)) - (np.min(x1) if side == "left" else np.max(x1))
    bottom0 = {c: np.max(y0[x0 == c]) for c in np.unique(x0)}
    top1 = {c + dx: np.min(y1[x1 == c]) for c in np.unique(x1)}
    
    dy = int(np.percentile([bottom0[c] - top1[c] + 1 for c in bottom0 if c in top1], 92))
    return merge_canvases(img0, img1, thresh0, thresh1, dx, dy)

def ad_stitch(img_base, img_patch):
    m_b = (cv2.cvtColor(img_base, cv2.COLOR_BGR2GRAY) > 15).astype(np.uint8) * 255
    m_p = (cv2.cvtColor(img_patch, cv2.COLOR_BGR2GRAY) > 15).astype(np.uint8) * 255
    
    y_p, x_p = np.where(cv2.Canny(m_p, 50, 150) == 255)
    dist_out = cv2.distanceTransform(~m_b, cv2.DIST_L2, 5)
    dist_in = cv2.distanceTransform(m_b, cv2.DIST_L2, 5)
    
    h_b, w_b = img_base.shape[:2]
    h_p, w_p = img_patch.shape[:2]
    best_dx, best_dy, min_score = 0, 0, float('inf')
    
    for dy in range(-int(h_b * 0.4), int(h_b * 0.8)):
        for dx in range(int(w_b * 0.1), int(w_b * 1.2)):
            sy, sx = y_p + dy, x_p + dx
            valid = (sy >= 0) & (sy < h_b) & (sx >= 0) & (sx < w_b)
            if np.sum(valid) < 15:
                continue
                
            vy, vx = sy[valid], sx[valid]
            ol = m_b[vy, vx] == 255
            ol_cnt = np.sum(ol)
            if ol_cnt / len(vy) > 0.35:
                continue
                
            n_vy, n_vx = vy[~ol], vx[~ol]
            if len(n_vy) == 0:
                continue
                
            gaps = dist_out[n_vy, n_vx]
            penalty = np.mean(dist_in[vy[ol], vx[ol]]) * 3.0 if ol_cnt > 0 else 0.0
            score = np.mean(gaps) + penalty - np.sum(gaps < 4.0) * 0.2
            
            if score < min_score:
                min_score, best_dx, best_dy = score, dx, dy

    oy, ox = max(0, -best_dy), max(0, -best_dx)
    mh, mw = max(h_b + oy, h_p + best_dy + oy), max(w_b + ox, w_p + best_dx + ox)
    
    canvas = np.zeros((mh, mw, 3), dtype=np.uint8)
    canvas[oy:oy+h_b, ox:ox+w_b] = img_base
    
    py, px = best_dy + oy, best_dx + ox
    canvas[py:py+h_p, px:px+w_p][m_p == 255] = img_patch[m_p == 255]
    return canvas



In [ ]:
L = isolated_pieces_list
canvas = L[0]
canvas = stitch_by_edge(canvas, L[1], ref="bottom")
canvas = stitch_by_edge(canvas, L[2], ref="bottom")
canvas = stitch_by_margin(L[4], canvas, side="left")

canvas75 = stitch_by_edge(L[7], L[5], ref="bottom")
canvas = stitch_by_margin(canvas75, canvas, side="left")

canvas63 = stitch_by_margin(L[6], L[3], side="right")
canvas638 = stitch_by_margin(L[8], canvas63, side="right")
canvas = stitch_by_edge(canvas, canvas638, ref="bottom")

canvas9_10 = stitch_by_edge(L[9], L[10], ref="bottom")
canvas = stitch_by_margin(canvas9_10, canvas, side="left")

canvas_12_11 = ad_stitch(L[12], L[11])
canvas = stitch_by_margin(canvas_12_11, canvas, side="left")
canvas = stitch_by_margin(L[15], canvas, side="left")

canvas_19_16 = ad_stitch(L[19], L[16])
canvas_13_14 = ad_stitch(L[13], L[14])

c1 = stitch_by_margin(L[17], canvas_13_14, side="right")
c2 = stitch_by_edge(canvas_19_16, L[20], ref="top")
c2 = stitch_by_edge(c2, L[18], ref="top")
c2 = stitch_by_edge(c2, c1, ref="top")

canvas = stitch_by_margin(c2, canvas, side="left")

plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
import cv2
import numpy as np

def ad_stitch(img_base, img_patch):
    m_b = (cv2.cvtColor(img_base, cv2.COLOR_BGR2GRAY) > 15).astype(np.uint8) * 255
    m_p = (cv2.cvtColor(img_patch, cv2.COLOR_BGR2GRAY) > 15).astype(np.uint8) * 255
    
    y_p, x_p = np.where(cv2.Canny(m_p, 50, 150) == 255)
    dist_out = cv2.distanceTransform(~m_b, cv2.DIST_L2, 5)
    dist_in = cv2.distanceTransform(m_b, cv2.DIST_L2, 5)
    
    h_b, w_b = img_base.shape[:2]
    h_p, w_p = img_patch.shape[:2]
    best_dx, best_dy, min_score = 0, 0, float('inf')
    
    for dy in range(-int(h_b * 0.8), int(h_b * 0.8)):
        for dx in range(-int(w_b * 0.8), int(w_b * 1.2)):
            sy, sx = y_p + dy, x_p + dx
            valid = (sy >= 0) & (sy < h_b) & (sx >= 0) & (sx < w_b)
            if np.sum(valid) < 15:
                continue
                
            vy, vx = sy[valid], sx[valid]
            ol = m_b[vy, vx] == 255
            ol_cnt = np.sum(ol)
            if ol_cnt / len(vy) > 0.35:
                continue
                
            n_vy, n_vx = vy[~ol], vx[~ol]
            if len(n_vy) == 0:
                continue
                
            gaps = dist_out[n_vy, n_vx]
            penalty = np.mean(dist_in[vy[ol], vx[ol]]) * 3.0 if ol_cnt > 0 else 0.0
            score = np.mean(gaps) + penalty - np.sum(gaps < 4.0) * 0.2
            
            if score < min_score:
                min_score, best_dx, best_dy = score, dx, dy

    oy, ox = max(0, -best_dy), max(0, -best_dx)
    mh, mw = max(h_b + oy, h_p + best_dy + oy), max(w_b + ox, w_p + best_dx + ox)
    
    canvas = np.zeros((mh, mw, 3), dtype=np.uint8)
    canvas[oy:oy+h_b, ox:ox+w_b] = img_base
    
    py, px = best_dy + oy, best_dx + ox
    canvas[py:py+h_p, px:px+w_p][m_p == 255] = img_patch[m_p == 255]
    return canvas

In [ ]:
def ad_stitch(img_base, img_patch):
   
    m_b = (cv2.cvtColor(img_base, cv2.COLOR_BGR2GRAY) > 15).astype(np.uint8) * 255
    m_p = (cv2.cvtColor(img_patch, cv2.COLOR_BGR2GRAY) > 15).astype(np.uint8) * 255
    
    y_p, x_p = np.where(cv2.Canny(m_p, 50, 150) == 255)
    dist_out = cv2.distanceTransform(~m_b, cv2.DIST_L2, 5)
    dist_in = cv2.distanceTransform(m_b, cv2.DIST_L2, 5)
    
    h_b, w_b = img_base.shape[:2]
    h_p, w_p = img_patch.shape[:2]
    best_dx, best_dy, min_score = 0, 0, float('inf')
    
    for dy in range(-int(h_b * 0.8), int(h_b * 0.8)):
        for dx in range(-int(w_b * 0.8), int(w_b * 1.2)):
            sy, sx = y_p + dy, x_p + dx
            valid = (sy >= 0) & (sy < h_b) & (sx >= 0) & (sx < w_b)
            if np.sum(valid) < 15:
                continue
                
            vy, vx = sy[valid], sx[valid]
            ol = m_b[vy, vx] == 255
            ol_cnt = np.sum(ol)
            if ol_cnt / len(vy) > 0.35:
                continue
                
            n_vy, n_vx = vy[~ol], vx[~ol]
            if len(n_vy) == 0:
                continue
                
            gaps = dist_out[n_vy, n_vx]
            penalty = np.mean(dist_in[vy[ol], vx[ol]]) * 3.0 if ol_cnt > 0 else 0.0
            score = np.mean(gaps) + penalty - np.sum(gaps < 4.0) * 0.2
            
            if score < min_score:
                min_score, best_dx, best_dy = score, dx, dy

    oy, ox = max(0, -best_dy), max(0, -best_dx)
    mh, mw = max(h_b + oy, h_p + best_dy + oy), max(w_b + ox, w_p + best_dx + ox)
    
    canvas = np.zeros((mh, mw, 3), dtype=np.uint8)
    canvas[oy:oy+h_b, ox:ox+w_b] = img_base
    
    py, px = best_dy + oy, best_dx + ox
    canvas[py:py+h_p, px:px+w_p][m_p == 255] = img_patch[m_p == 255]
    return canvas

canvas = isolated_pieces_list[0]
for i in range(1, len(isolated_pieces_list)):
    print(f"Stitching piece {i + 1}/{len(isolated_pieces_list)}...")
    canvas = ad_stitch(canvas, isolated_pieces_list[i])
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.show()


In [ ]:
plt.imshow(cv2.cvtColor(isolated_pieces_list[10], cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
a1=ad_stitch(isolated_pieces_list[10], isolated_pieces_list[13])
plt.imshow(cv2.cvtColor(a1, cv2.COLOR_BGR2RGB))
plt.show()